In [1]:
#r "nuget: Microsoft.Extensions.Configuration.Json, 10.0.13"
#r "nuget: OpenAI, 2.14.0"
#r "nuget: HtmlAgilityPack, 1.13.1"

using System;
using System.ClientModel;
using System.Collections.Generic;
using System.IO;
using System.Linq;
using System.Net.Http;
using System.Text.Json;
using System.Threading.Tasks;
using HtmlAgilityPack;
using Microsoft.Extensions.Configuration;
using OpenAI;
using OpenAI.Chat;

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages HtmlAgilityPack, 1.13.1 Microsoft.Extensions.Configuration.Json, 10.0.13 OpenAI, 2.14.0

In [2]:
// Check key (start the script/REPL in the 01_Maze folder so the appsettings files are found)
Console.WriteLine("\nCHECKING OPENROUTER API KEY\n");

var config = new ConfigurationBuilder()
    .SetBasePath(Directory.GetCurrentDirectory())
    .AddJsonFile("appsettings.json")
    .AddJsonFile("appsettings.local.json", optional: true)
    .Build();

var openrouterApiKey = config["Settings:OpenRouterApiKey"];

if (string.IsNullOrEmpty(openrouterApiKey))
{
    throw new InvalidOperationException("No OpenRouter API key found. Please set the 'OpenRouterApiKey' in appsettings.local.json.");
}

Console.WriteLine($"OpenRouter API key exists and begins {openrouterApiKey[..9]}\n");


CHECKING OPENROUTER API KEY

OpenRouter API key exists and begins sk-or-v1-



In [3]:
// Connect to OpenRouter

const string OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1";
const string MODEL = "openrouter/free";

OpenAIClient client = new(new ApiKeyCredential(openrouterApiKey), new OpenAIClientOptions
{
    Endpoint = new Uri(OPENROUTER_BASE_URL)
});

var chatClient = client.GetChatClient(MODEL);

In [4]:
// Hello test

var message = new UserChatMessage("Hello! How are you today?");
var completion = await chatClient.CompleteChatAsync(message);

Console.WriteLine($"\nResponse from OpenRouter: {completion.Value.Content[0].Text}");

Error: System.ClientModel.ClientResultException: HTTP 429 (: )

Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day
   at OpenAI.ClientPipelineExtensions.ProcessMessageAsync(ClientPipeline pipeline, PipelineMessage message, RequestOptions options)
   at OpenAI.Chat.ChatClient.CompleteChatAsync(BinaryContent content, RequestOptions options)
   at OpenAI.Chat.ChatClient.CompleteChatAsync(IEnumerable`1 messages, ChatCompletionOptions options, RequestOptions requestOptions)
   at OpenAI.Chat.ChatClient.CompleteChatAsync(ChatMessage[] messages)
   at Submission#5.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [ ]:
// Part 1 of the harness: the tool

const string MAZE_URL = "https://maze.edwarddonner.com";
var httpClient = new HttpClient { Timeout = TimeSpan.FromSeconds(20) };

async Task<string> ReadWebPage(string url)
{
    Console.WriteLine($"\nReading web page: {url}");
    if (!url.StartsWith(MAZE_URL))
    {
        return $"Sorry, this tool can only read pages in the maze. Please ask for a URL that starts with {MAZE_URL}";
    }

    var html = await httpClient.GetStringAsync(url);
    var page = new HtmlDocument();
    page.LoadHtml(html);

    var body = page.DocumentNode.SelectSingleNode("//body") ?? page.DocumentNode;
    var lines = body
        .DescendantsAndSelf()
        .OfType<HtmlTextNode>()
        .Where(node => node.ParentNode?.Name is not ("script" or "style"))
        .Select(node => HtmlEntity.DeEntitize(node.Text).Trim())
        .Where(text => text.Length > 0);

    return string.Join("\n", lines);
}

In [ ]:
// Reading web pages

Console.WriteLine("\nREADING WEB PAGES");
Console.WriteLine($"\n{await ReadWebPage("https://maze.edwarddonner.com/")}");

Console.WriteLine($"\n{await ReadWebPage("https://nowakdawid.pl/")}");

In [ ]:
// The tool for a ChatClient

var readWebPageTool = ChatTool.CreateFunctionTool(
    functionName: "read_web_page",
    functionDescription: "Read a web page and return the text on the page",
    functionParameters: BinaryData.FromString("""
        {
            "type": "object",
            "properties": {
                "url": { "type": "string", "description": "The full URL of the web page to request" }
            },
            "required": ["url"]
        }
        """));

var chatOptions = new ChatCompletionOptions
{
    Tools = { readWebPageTool }
};

In [ ]:
// Handle tool call

async Task<ToolChatMessage> HandleToolCall(ChatToolCall toolCall)
{
    using var arguments = JsonDocument.Parse(toolCall.FunctionArguments.ToString());
    var url = arguments.RootElement.GetProperty("url").GetString() ?? "";
    var result = await ReadWebPage(url);
    return new ToolChatMessage(toolCall.Id, result);
}

In [ ]:
// Using the tool once
Console.WriteLine("\nUSING THE TOOL ONCE");

List<ChatMessage> messages =
[
    new UserChatMessage("Please read https://maze.edwarddonner.com and tell me what it says. Just describe the page for now, without going into the maze.")
];

ChatCompletion response = await chatClient.CompleteChatAsync(messages, chatOptions);

Console.WriteLine($"\nAnswered by {response.Model}");
Console.WriteLine($"Finish reason: {response.FinishReason}");
foreach (var toolCall in response.ToolCalls)
{
    Console.WriteLine($"Tool call {toolCall.Id}: {toolCall.FunctionName}({toolCall.FunctionArguments})");
}

In [ ]:
// Send the tool results back

messages.Add(new AssistantChatMessage(response));
foreach (var toolCall in response.ToolCalls)
{
    messages.Add(await HandleToolCall(toolCall));
}

response = await chatClient.CompleteChatAsync(messages, chatOptions);

Console.WriteLine($"\nAnswered by {response.Model}");
Console.WriteLine(response.Content[0].Text);


In [ ]:
// Part 2 of the harness: the loop

const string SYSTEM_PROMPT = """
You are an agent that completes tasks by using your tools, one step at a time.
Put any map or diagram in a formatted way so that it lines up. Do not use emojis.
""";

async Task RunAgent(string task, int maxSteps = 25)
{
    List<ChatMessage> messages =
    [
        new SystemChatMessage(SYSTEM_PROMPT),
        new UserChatMessage(task)
    ];

    for (int i = 0; i < maxSteps; i++)
    {
        ChatCompletion response = await chatClient.CompleteChatAsync(messages, chatOptions);
        messages.Add(new AssistantChatMessage(response));
        Console.WriteLine($"Step {i + 1}: Answered by {response.Model}");

        if (response.FinishReason == ChatFinishReason.ToolCalls)
        {
            foreach (var toolCall in response.ToolCalls)
            {
                messages.Add(await HandleToolCall(toolCall));
            }
        }
        else if (response.FinishReason == ChatFinishReason.Stop)
        {
            Console.WriteLine($"\nThe agent completed the task successfully.");
            Console.WriteLine($"\nFinal answer: {response.Content[0].Text}");
            return;
        }
    }

    Console.WriteLine($"\nThe agent ran out of steps.");
}

In [ ]:
// Into the maze
Console.WriteLine("\nINTO THE MAZE");

await RunAgent("Have a look at https://maze.edwarddonner.com and do what it says.");